# Lab 3 - TinySRCNNResidual (No-Add Colab A100, latest resume LR floor)

- Goal: resume the latest TinySRCNN no-add checkpoint and continue toward epoch `10000`.
- Expected input/output: `256x256x3`.
- Resume checkpoint: latest matching `runs/**/checkpoints/last.pth`.


## 1. Setup

This notebook follows the Lab 3 rubric structure while keeping the implementation lean.


In [1]:
try:
    import google.colab  # type: ignore
    IN_COLAB_BOOTSTRAP = True
except Exception:
    IN_COLAB_BOOTSTRAP = False

if IN_COLAB_BOOTSTRAP:
    %pip install -q onnx onnxruntime

import csv
import json
import math
import os
import random
import sys
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import onnx
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
DRIVE_ROOT = Path('/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3')
DRIVE_DATA_ROOT = DRIVE_ROOT / 'Data'
DRIVE_RUNS_ROOT = DRIVE_ROOT / 'runs'

try:
    BICUBIC_RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    BICUBIC_RESAMPLE = Image.BICUBIC

try:
    FLIP_LEFT_RIGHT = Image.Transpose.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.Transpose.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.Transpose.ROTATE_90
    ROTATE_180 = Image.Transpose.ROTATE_180
    ROTATE_270 = Image.Transpose.ROTATE_270
except AttributeError:
    FLIP_LEFT_RIGHT = Image.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.ROTATE_90
    ROTATE_180 = Image.ROTATE_180
    ROTATE_270 = Image.ROTATE_270


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def now_stamp() -> str:
    return datetime.now().strftime('%H%M%S_%d%m')


def is_colab_runtime() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def auto_num_workers(in_colab: bool) -> int:
    if sys.platform == 'darwin':
        return 0
    cpu_count = max(os.cpu_count() or 2, 2)
    if in_colab:
        return min(4, max(cpu_count - 1, 2))
    return min(2, cpu_count)


def resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'Data').exists() and (candidate / 'runs').exists():
            return candidate
    for candidate in [current, *current.parents]:
        if (candidate / 'Data').exists():
            return candidate
    raise FileNotFoundError(f'Could not locate repo root from {start.resolve()}')


def make_unique_run_root(runs_root: Path, run_slug: str = '') -> Path:
    safe_slug = ''.join(ch if ch.isalnum() or ch in '_-' else '_' for ch in run_slug).strip('_')
    suffix = f'_{safe_slug}' if safe_slug else ''
    candidate = runs_root / f'{now_stamp()}{suffix}'
    while candidate.exists():
        time.sleep(1.1)
        candidate = runs_root / f'{now_stamp()}{suffix}'
    return candidate


def read_checkpoint_identity(checkpoint_path: Path) -> dict[str, Any] | None:
    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
    except Exception:
        return None
    config = checkpoint.get('config') or {}
    model_id = config.get('model_id')
    if not model_id:
        return None
    return {
        'path': str(checkpoint_path),
        'model_id': model_id,
        'epoch': int(checkpoint.get('epoch', 0)),
        'kind': checkpoint_path.name,
        'mtime': checkpoint_path.stat().st_mtime,
    }


def find_latest_model_checkpoint(runs_root: Path, model_id: str, filename: str = 'last.pth') -> str | None:
    candidates: list[dict[str, Any]] = []
    for checkpoint_path in runs_root.rglob(f'checkpoints/{filename}'):
        identity = read_checkpoint_identity(checkpoint_path)
        if identity and identity['model_id'] == model_id:
            candidates.append(identity)
    if not candidates:
        return None
    winner = max(candidates, key=lambda item: item['mtime'])
    return winner['path']


def resolve_resume_checkpoint_path(resume_checkpoint_path: str | None, runs_root: Path, model_id: str) -> str | None:
    if not resume_checkpoint_path:
        return None
    if resume_checkpoint_path == 'last.pth':
        return find_latest_model_checkpoint(runs_root, model_id, filename='last.pth')
    path = Path(resume_checkpoint_path)
    if not path.is_absolute():
        path = runs_root / path
    if path.name == 'exports':
        candidate_paths = [path.parent / 'checkpoints' / 'last.pth']
    elif path.is_dir():
        candidate_paths = [path / 'last.pth']
    else:
        candidate_paths = [path]
    for candidate in candidate_paths:
        if candidate.is_file():
            return str(candidate)
    return None



@dataclass
class RunConfig:
    model_id: str = 'tinysrcnn_noadd_lrelu'
    run_slug: str = 'resume_latest_lr_floor'
    enable_resume: bool = True
    resume_checkpoint_path: str | None = 'last.pth'
    seed: int = 1337
    train_patch_size: int = 256
    eval_size: int = 256
    batch_size: int = 16
    epochs: int = 10000
    lr: float = 1e-4
    min_lr: float = 1e-5
    weight_decay: float = 1e-4
    warmup_epochs: int = 8
    num_workers: int | None = 2
    prefetch_factor: int = 2
    use_amp: bool = True
    use_ema: bool = True
    ema_decay: float = 0.999


cfg = RunConfig()
in_colab = is_colab_runtime()
if in_colab:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)
    data_root = DRIVE_DATA_ROOT
    runs_root = DRIVE_RUNS_ROOT
    if not data_root.exists():
        raise FileNotFoundError(f'Google Drive data root not found: {data_root}')
else:
    repo_root = resolve_repo_root(Path.cwd())
    data_root = repo_root / 'Data'
    runs_root = repo_root / 'runs'
    if not data_root.exists():
        raise FileNotFoundError(f'Data root not found: {data_root}')

runs_root.mkdir(parents=True, exist_ok=True)
if cfg.num_workers is None:
    cfg.num_workers = auto_num_workers(in_colab)
if cfg.enable_resume:
    cfg.resume_checkpoint_path = resolve_resume_checkpoint_path(cfg.resume_checkpoint_path, runs_root, cfg.model_id)
    if cfg.resume_checkpoint_path is None:
        raise FileNotFoundError(f'No latest last.pth found for model_id={cfg.model_id!r} under {runs_root}')
else:
    cfg.resume_checkpoint_path = None

run_root = make_unique_run_root(runs_root, cfg.run_slug)
checkpoints_dir = run_root / 'checkpoints'
exports_dir = run_root / 'exports'
preview_dir = exports_dir / 'val_preview'
summary_path = run_root / 'summary.json'
metrics_csv_path = run_root / 'metrics.csv'

for directory in [run_root, checkpoints_dir, exports_dir, preview_dir]:
    directory.mkdir(parents=True, exist_ok=True)

set_global_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, 'allow_tf32'):
        torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')

print({
    'device': str(device),
    'data_root': str(data_root),
    'runs_root': str(runs_root),
    'run_root': str(run_root),
    'config': asdict(cfg),
})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'device': 'cuda', 'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'runs_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs', 'run_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor', 'config': {'model_id': 'tinysrcnn_noadd_lrelu', 'run_slug': 'resume_latest_lr_floor', 'enable_resume': True, 'resume_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/191346_2604_resume_5144_to_10000/checkpoints/last.pth', 'seed': 1337, 'train_patch_size': 256, 'eval_size': 256, 'batch_size': 16, 'epochs': 10000, 'lr': 0.0001, 'min_lr': 1e-05, 'weight_decay': 0.0001, 'warmup_epochs': 8, 'num_workers': 2, 'prefetch_factor': 2, 'use_amp': True, 'use_ema': True, 'ema_decay': 0.999}}


## 2. Data


In [2]:
L3_TRAIN_SPLITS = (
    ('LR_HR/L3_HR_train1', 'L3_LR/L3_L3_train1', 'L3_HR_train1'),
    ('LR_HR/L3_HR_train2', 'L3_LR/L3_L3_train2', 'L3_HR_train2'),
    ('LR_HR/L3_HR_train3', 'L3_LR/L3_LR_train3', 'L3_HR_train3'),
    ('LR_HR/L3_HR_train4', 'L3_LR/L3_LR_train4', 'L3_HR_train4'),
    ('LR_HR/L3_HR_train', 'L3_LR/L3_LR_train', 'L3_HR_train'),
)
L3_VAL_DIRS = ('L3_HR_valid', 'L3_LR_valid')


def scan_image_directory(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        raise FileNotFoundError(f'Missing directory: {directory}')
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    }


def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
    hr_map = scan_image_directory(hr_dir)
    lr_map = scan_image_directory(lr_dir)
    pairs: list[tuple[Path, Path, str]] = []
    for stem in sorted(set(hr_map) & set(lr_map)):
        name = stem if split_name == 'val' else f'{split_name}/{stem}'
        pairs.append((lr_map[stem], hr_map[stem], name))
    return pairs


def collect_all_pairs(data_root: Path) -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
    train_pairs: list[tuple[Path, Path, str]] = []
    for hr_rel, lr_rel, split_name in L3_TRAIN_SPLITS:
        train_pairs.extend(collect_split_pairs(data_root / hr_rel, data_root / lr_rel, split_name))
    val_pairs = collect_split_pairs(data_root / L3_VAL_DIRS[0], data_root / L3_VAL_DIRS[1], 'val')
    return train_pairs, val_pairs


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], *, train: bool, patch_size: int, eval_size: int, seed: int):
        self.pairs = pairs
        self.train = train
        self.patch_size = patch_size
        self.eval_size = eval_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    @staticmethod
    def to_tensor(img: Image.Image) -> torch.Tensor:
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

    def augment_pair(self, lr: Image.Image, hr: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
        rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
        width, height = lr.size
        patch = self.patch_size
        if width < patch or height < patch:
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            width, height = lr.size
        left = rng.randint(0, width - patch)
        top = rng.randint(0, height - patch)
        box = (left, top, left + patch, top + patch)
        lr = lr.crop(box)
        hr = hr.crop(box)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_LEFT_RIGHT)
            hr = hr.transpose(FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_TOP_BOTTOM)
            hr = hr.transpose(FLIP_TOP_BOTTOM)
        rot = rng.choice([0, 1, 2, 3])
        if rot:
            rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
            lr = lr.transpose(rotate_ops[rot - 1])
            hr = hr.transpose(rotate_ops[rot - 1])
        return lr, hr

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        lr_path, hr_path, name = self.pairs[index]
        lr = Image.open(lr_path).convert('RGB')
        hr = Image.open(hr_path).convert('RGB')
        if self.train:
            lr, hr = self.augment_pair(lr, hr, index)
        elif lr.size != (self.eval_size, self.eval_size):
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
        return {'lr': self.to_tensor(lr), 'hr': self.to_tensor(hr), 'name': name}


train_pairs, val_pairs = collect_all_pairs(data_root)
if not train_pairs:
    raise FileNotFoundError(f'No training pairs found under {data_root}')
if not val_pairs:
    raise FileNotFoundError(f'No validation pairs found under {data_root}')
print({
    'data_root': str(data_root),
    'train_pairs': len(train_pairs),
    'val_pairs': len(val_pairs),
})

train_ds = PairedSRDataset(train_pairs, train=True, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)
val_ds = PairedSRDataset(val_pairs, train=False, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)

loader_num_workers = cfg.num_workers if cfg.num_workers is not None else auto_num_workers(in_colab)
train_loader_kwargs = {
    'dataset': train_ds,
    'batch_size': cfg.batch_size,
    'shuffle': True,
    'num_workers': loader_num_workers,
    'pin_memory': device.type == 'cuda',
}
val_loader_kwargs = {
    'dataset': val_ds,
    'batch_size': 1,
    'shuffle': False,
    'num_workers': loader_num_workers,
    'pin_memory': device.type == 'cuda',
}
if loader_num_workers > 0:
    train_loader_kwargs['prefetch_factor'] = cfg.prefetch_factor
    val_loader_kwargs['prefetch_factor'] = cfg.prefetch_factor

train_loader = DataLoader(**train_loader_kwargs)
val_loader = DataLoader(**val_loader_kwargs)

sample_batch = next(iter(val_loader))
print({
    'validation_input_shape': tuple(sample_batch['lr'].shape),
    'validation_target_shape': tuple(sample_batch['hr'].shape),
    'sample_name': sample_batch['name'][0],
})


{'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'train_pairs': 2217, 'val_pairs': 110}
{'validation_input_shape': (1, 3, 256, 256), 'validation_target_shape': (1, 3, 256, 256), 'sample_name': '0000'}


## 3. Model and Training


In [3]:
class TinySRCNNNoAdd(nn.Module):
    def __init__(self, channels: int = 64, slope: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1),
            nn.LeakyReLU(slope, inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.LeakyReLU(slope, inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.LeakyReLU(slope, inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.LeakyReLU(slope, inplace=False),
            nn.Conv2d(channels, 3, 3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def count_parameters(module: nn.Module) -> int:
    return int(sum(param.numel() for param in module.parameters()))


class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for key, value in model.state_dict().items():
            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow, strict=True)


def make_grad_scaler(enabled: bool):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(device_type: str, enabled: bool):
    if not enabled:
        return nullcontext()
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


@torch.no_grad()
def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=(1, 2, 3))
    psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=1e-12))
    return float(psnr.mean().item())


def compute_loss(pred: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
    loss_mse = F.mse_loss(pred, hr)
    loss_l1 = F.l1_loss(pred, hr)
    return 0.8 * loss_mse + 0.2 * loss_l1


def lr_at_epoch(epoch_idx: int, *, base_lr: float, min_lr: float, warmup_epochs: int, total_epochs: int) -> float:
    step = epoch_idx + 1
    if warmup_epochs > 0 and step <= warmup_epochs:
        return base_lr * step / max(1, warmup_epochs)
    progress = max(0.0, (step - warmup_epochs) / max(1, total_epochs - warmup_epochs))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return min_lr + cosine * (base_lr - min_lr)


@torch.inference_mode()
def evaluate_psnr(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    meter = 0.0
    count = 0
    for batch in loader:
        lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
        hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))
        pred = model(lr).clamp(0.0, 1.0)
        meter += batch_psnr(pred, hr)
        count += 1
    return meter / max(count, 1)


@torch.inference_mode()
def compute_input_psnr(loader: DataLoader, device: torch.device) -> float:
    meter = 0.0
    count = 0
    for batch in loader:
        lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
        hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))
        meter += batch_psnr(lr, hr)
        count += 1
    return meter / max(count, 1)


def tensor_to_uint8_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().cpu().clamp(0.0, 1.0)
    arr = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)


@torch.no_grad()
def write_val_preview(model: nn.Module, loader: DataLoader, out_dir: Path, device: torch.device) -> dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    batch = next(iter(loader))
    lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
    hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))
    pred = model(lr).clamp(0.0, 1.0)
    name = batch['name'][0]
    lr_path = out_dir / 'val_preview_input.png'
    pred_path = out_dir / 'val_preview_pred.png'
    hr_path = out_dir / 'val_preview_target.png'
    tensor_to_uint8_image(lr[0]).save(lr_path)
    tensor_to_uint8_image(pred[0]).save(pred_path)
    tensor_to_uint8_image(hr[0]).save(hr_path)
    return {
        'sample_name': str(name),
        'input': str(lr_path),
        'prediction': str(pred_path),
        'target': str(hr_path),
    }


def checkpoint_payload(model: nn.Module, ema: EMA | None, optimizer: torch.optim.Optimizer, epoch: int, best_val_psnr: float, row: dict[str, Any]) -> dict[str, Any]:
    return {
        'config': asdict(cfg),
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema.shadow if ema is not None else None,
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_psnr': float(best_val_psnr),
        'selected_weight_source': 'ema' if ema is not None else 'raw',
        'last_metrics': row,
    }


def build_eval_model_from_checkpoint(checkpoint_path: Path, device: torch.device) -> nn.Module:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = TinySRCNNNoAdd().to(device)
    selected = checkpoint.get('selected_weight_source', 'raw')
    ema_state = checkpoint.get('ema_state_dict')
    if selected == 'ema' and ema_state is not None:
        model.load_state_dict(ema_state, strict=True)
    else:
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    model.eval()
    return model


model = TinySRCNNNoAdd().to(device)
eval_model = TinySRCNNNoAdd().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = make_grad_scaler(enabled=(cfg.use_amp and device.type == 'cuda'))
ema = EMA(model, decay=cfg.ema_decay) if cfg.use_ema else None
start_epoch = 0
best_val_psnr = float('-inf')
if cfg.enable_resume and cfg.resume_checkpoint_path:
    checkpoint = torch.load(cfg.resume_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    optimizer_state = checkpoint.get('optimizer_state_dict')
    if optimizer_state is not None:
        optimizer.load_state_dict(optimizer_state)
    ema_state = checkpoint.get('ema_state_dict')
    if ema is not None and ema_state is not None:
        ema.shadow = {key: value.detach().clone() for key, value in ema_state.items()}
    start_epoch = int(checkpoint.get('epoch', 0))
    best_val_psnr = float(checkpoint.get('best_val_psnr', float('-inf')))
if ema is not None:
    ema.copy_to(eval_model)
else:
    eval_model.load_state_dict(model.state_dict())

model_parameter_count = count_parameters(model)
print({
    'model_id': cfg.model_id,
    'architecture': 'Conv3x3(3->64) x4 with LeakyReLU, final Conv3x3(64->3), no Add',
    'params': model_parameter_count,
    'resume_checkpoint_path': cfg.resume_checkpoint_path,
    'start_epoch': start_epoch,
})

input_psnr = compute_input_psnr(val_loader, device)
print({'baseline_input_psnr': input_psnr})


{'model_id': 'tinysrcnn_noadd_lrelu', 'architecture': 'Conv3x3(3->64) x4 with LeakyReLU, final Conv3x3(64->3), no Add', 'params': 114307, 'resume_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/191346_2604_resume_5144_to_10000/checkpoints/last.pth', 'start_epoch': 8974}
{'baseline_input_psnr': 23.133060299266468}


In [4]:
metrics_header = [
    'epoch',
    'epoch_time_sec',
    'lr',
    'train_loss',
    'train_psnr',
    'val_psnr',
    'best_val_psnr',
]

best_ckpt_path = checkpoints_dir / 'best.pth'
last_ckpt_path = checkpoints_dir / 'last.pth'

with metrics_csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=metrics_header)
    writer.writeheader()

for epoch in range(start_epoch, cfg.epochs):
    train_ds.set_epoch(epoch)
    epoch_lr = lr_at_epoch(
        epoch,
        base_lr=cfg.lr,
        min_lr=cfg.min_lr,
        warmup_epochs=cfg.warmup_epochs,
        total_epochs=cfg.epochs,
    )
    for group in optimizer.param_groups:
        group['lr'] = epoch_lr

    epoch_start = time.time()
    model.train()
    train_loss_sum = 0.0
    train_psnr_sum = 0.0
    train_batches = 0

    for batch in train_loader:
        lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
        hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))

        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device.type, enabled=(cfg.use_amp and device.type == 'cuda')):
            pred = model(lr)
            loss = compute_loss(pred, hr)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if ema is not None:
            ema.update(model)

        train_loss_sum += float(loss.item())
        train_psnr_sum += batch_psnr(pred.detach().clamp(0.0, 1.0), hr)
        train_batches += 1

    if ema is not None:
        ema.copy_to(eval_model)
    else:
        eval_model.load_state_dict(model.state_dict())
    eval_model.eval()

    avg_train_loss = train_loss_sum / max(train_batches, 1)
    avg_train_psnr = train_psnr_sum / max(train_batches, 1)
    val_psnr = evaluate_psnr(eval_model, val_loader, device)
    epoch_time = time.time() - epoch_start
    is_best = val_psnr > best_val_psnr
    if is_best:
        best_val_psnr = val_psnr

    row = {
        'epoch': epoch + 1,
        'epoch_time_sec': round(epoch_time, 2),
        'lr': epoch_lr,
        'train_loss': round(avg_train_loss, 6),
        'train_psnr': round(avg_train_psnr, 4),
        'val_psnr': round(val_psnr, 4),
        'best_val_psnr': round(best_val_psnr, 4),
    }
    print(row)

    with metrics_csv_path.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=metrics_header)
        writer.writerow(row)

    payload = checkpoint_payload(model, ema, optimizer, epoch + 1, best_val_psnr, row)
    torch.save(payload, last_ckpt_path)
    if is_best:
        torch.save(payload, best_ckpt_path)


{'epoch': 8975, 'epoch_time_sec': 245.92, 'lr': 1.231666434978453e-05, 'train_loss': 0.011788, 'train_psnr': 25.033, 'val_psnr': 23.8963, 'best_val_psnr': 23.8965}
{'epoch': 8976, 'epoch_time_sec': 21.83, 'lr': 1.231218532963957e-05, 'train_loss': 0.011775, 'train_psnr': 25.0394, 'val_psnr': 23.8963, 'best_val_psnr': 23.8965}
{'epoch': 8977, 'epoch_time_sec': 21.07, 'lr': 1.2307710529362104e-05, 'train_loss': 0.011794, 'train_psnr': 25.0292, 'val_psnr': 23.8962, 'best_val_psnr': 23.8965}
{'epoch': 8978, 'epoch_time_sec': 21.83, 'lr': 1.2303239949394474e-05, 'train_loss': 0.011786, 'train_psnr': 25.0348, 'val_psnr': 23.8961, 'best_val_psnr': 23.8965}
{'epoch': 8979, 'epoch_time_sec': 20.96, 'lr': 1.2298773590178623e-05, 'train_loss': 0.011774, 'train_psnr': 25.039, 'val_psnr': 23.8961, 'best_val_psnr': 23.8965}
{'epoch': 8980, 'epoch_time_sec': 21.71, 'lr': 1.2294311452156062e-05, 'train_loss': 0.011782, 'train_psnr': 25.0358, 'val_psnr': 23.896, 'best_val_psnr': 23.8965}
{'epoch': 8981

## 4. Validation


In [5]:
best_model = build_eval_model_from_checkpoint(best_ckpt_path, device)
preview_info = write_val_preview(best_model, val_loader, preview_dir, device)
best_val_psnr = evaluate_psnr(best_model, val_loader, device)
validation_summary = {
    'input_psnr': input_psnr,
    'val_psnr': best_val_psnr,
    'delta_psnr': best_val_psnr - input_psnr,
}
print({
    'validation_summary': validation_summary,
    'preview': preview_info,
    'best_checkpoint_path': str(best_ckpt_path),
})


{'validation_summary': {'input_psnr': 23.133060299266468, 'val_psnr': 23.89888187755238, 'delta_psnr': 0.7658215782859124}, 'preview': {'sample_name': '0000', 'input': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/exports/val_preview/val_preview_input.png', 'prediction': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/exports/val_preview/val_preview_pred.png', 'target': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/exports/val_preview/val_preview_target.png'}, 'best_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/checkpoints/best.pth'}


## 5. Export and Submission Artifacts

ONNX export is included here. MXQ conversion stays in a separate workflow.


In [6]:
onnx_path = exports_dir / f'{cfg.model_id}.onnx'
export_model = build_eval_model_from_checkpoint(best_ckpt_path, device)
dummy = torch.randn(1, 3, 256, 256, device=device)

torch.onnx.export(
    export_model,
    dummy,
    str(onnx_path),
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    dynamo=False,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
node_ops = [node.op_type for node in onnx_model.graph.node]
op_histogram = {op: node_ops.count(op) for op in sorted(set(node_ops))}
print({
    'onnx_path': str(onnx_path),
    'graph_op_count': len(node_ops),
    'op_histogram': op_histogram,
})

parity_result = {
    'attempted': False,
    'onnxruntime_available': False,
    'max_abs_diff': None,
}
try:
    import onnxruntime as ort

    parity_result['attempted'] = True
    parity_result['onnxruntime_available'] = True
    cpu_model = build_eval_model_from_checkpoint(best_ckpt_path, torch.device('cpu')).cpu().eval()
    parity_input = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        torch_out = cpu_model(parity_input).detach().cpu().numpy()
    ort_session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    ort_out = ort_session.run(['output'], {'input': parity_input.numpy()})[0]
    parity_result['max_abs_diff'] = float(np.max(np.abs(torch_out - ort_out)))
except ImportError:
    print('onnxruntime not installed; skipping CPU parity check')
print(parity_result)

summary = {
    'model_id': cfg.model_id,
    'run_root': str(run_root),
    'best_checkpoint_path': str(best_ckpt_path),
    'last_checkpoint_path': str(last_ckpt_path),
    'onnx_path': str(onnx_path),
    'metrics_csv_path': str(metrics_csv_path),
    'validation': validation_summary,
    'preview': preview_info,
    'onnx_parity': parity_result,
    'mxq_scope': 'Not handled in this notebook; separate workflow.',
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(summary)


/tmp/ipykernel_10513/1798986534.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


{'onnx_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/exports/tinysrcnn_noadd_lrelu.onnx', 'graph_op_count': 9, 'op_histogram': {'Conv': 5, 'LeakyRelu': 4}}
{'attempted': True, 'onnxruntime_available': True, 'max_abs_diff': 1.1920928955078125e-06}
{'model_id': 'tinysrcnn_noadd_lrelu', 'run_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor', 'best_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/checkpoints/best.pth', 'last_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/checkpoints/last.pth', 'onnx_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/180446_2704_resume_latest_lr_floor/exports/tinysrcnn_noadd_lrelu.onnx', 'metrics_csv_path': '/content/drive/MyDrive/Data 255 Class Spring 